# 02 - Mentions over time (per ticker)

**What this does:** reads the cleaned slice saved by **notebook 01**
(`data/processed/posts_slice.parquet`) — so it automatically uses the SAME
time window and subreddits you set there — finds the stock tickers in each
post, and counts mentions per day, **per source** (Reddit vs X) and
**combined**.

**Raw counts only - the upvote-weighted (score²) signal was REMOVED**
(2026-07-06): archived dumps carry each post's FINAL score, the upvotes it
collected days or weeks AFTER posting - weighting day-t mentions by final
scores reads the future, a look-ahead leak that would silently inflate
any backtest. Raw counts are immune: one post = 1 the moment it exists.

It saves TWO files: `daily_ticker_counts.parquet` (combined - notebook 03
and everything downstream use this) and
`daily_ticker_counts_by_source.parquet` (per-source - powers the
Reddit-vs-X comparison at the bottom). The chain is: **01 (slice +
screening) → 02 (counts) → 03 (take-offs)** — to change the window or
subreddits, edit notebook 01 and re-run the chain from there.

_Note:_ the first run downloads the official Nasdaq ticker list (needs internet)
and caches it under `data/reference/`. The universe now includes a
hand-curated DELISTED supplement (BBBY, WISH, SPRT...) so dead retail
favourites still count - see `src/ticker_universe.py` (survivorship fix).

**Ticker matching rules** (`src/extract_tickers.py`): a mention is either a
cashtag (`$GME`, any case) or a bare **ALL-CAPS** 4-5 letter word (`NVDA`)
that is in the ticker universe and not on the stop lists. Lowercase words
never count - "i took out a loan" is not a $LOAN mention. Word-tickers
(LOAN, EDGE, RENT, ... — classified by notebook 01's screening) only count
when written as cashtags.

In [1]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: C:\Users\alexd\Desktop\GIC\RetailFlow1


In [2]:
# ============ PARAMETERS - edit these ============
# NOTE: no TIME WINDOW here any more. The window and subreddits come from
# notebook 01, via the slice file it saves - one source of truth for the chain.
SLICE_PATH        = os.path.join(ROOT, 'data', 'processed', 'posts_slice.parquet')
CASHTAGS_ONLY     = False   # True = only count $TICKER (cleaner, fewer false hits)
TICKERS_TO_PLOT   = []      # e.g. ['GME', 'AMC']; [] = use the TOP_N most mentioned
TOP_N             = 6
DAILY_COUNTS_OUT  = os.path.join(ROOT, 'data', 'processed', 'daily_ticker_counts.parquet')
BY_SOURCE_OUT     = os.path.join(ROOT, 'data', 'processed', 'daily_ticker_counts_by_source.parquet')
# ==================================================

In [3]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Force-reload src modules so a kernel that imported them BEFORE notebook 01
# (re)built ticker_classification.csv picks up the fresh screening list.
import importlib
import src.extract_tickers, src.build_mentions
importlib.reload(src.extract_tickers)
importlib.reload(src.build_mentions)
from src.extract_tickers import SCREENED_STOP
from src.build_mentions import build_daily_counts
from src.ticker_universe import load_us_ticker_universe

if not os.path.exists(SLICE_PATH):
    raise FileNotFoundError(
        'posts_slice.parquet not found - run notebook 01 first '
        '(it saves the cleaned slice this notebook reads).')

cols = ['date', 'title', 'selftext']
if 'source' in pq.ParquetFile(SLICE_PATH).schema_arrow.names:
    cols.append('source')
posts = pd.read_parquet(SLICE_PATH, columns=cols)
if 'source' not in posts.columns:
    posts['source'] = 'reddit'   # X data not merged yet - see add_x_data.py
print('posts:', f'{len(posts):,}', '| window:', posts['date'].min(), '->', posts['date'].max())
print('by source:', posts['source'].value_counts().to_dict())

if SCREENED_STOP:
    print(len(SCREENED_STOP), 'word-tickers demoted to cashtag-only (LOAN, EDGE, RENT, ...)')
else:
    print('WARNING: ticker_classification.csv not found - word-tickers like')
    print('LOAN/EDGE will pollute the counts. Run notebook 01, last section.')

# Valid US tickers (cached; incl. the delisted supplement - survivorship fix).
universe = load_us_ticker_universe(Path(ROOT) / 'data' / 'reference', max_cache_age_days=365)
print('ticker universe size:', len(universe))

# Count mentions PER SOURCE first, then sum into the combined signal.
parts = []
for src_name in sorted(posts['source'].unique()):
    d = build_daily_counts(posts[posts['source'] == src_name], universe,
                           cashtags_only=CASHTAGS_ONLY)
    d['source'] = src_name
    parts.append(d)
by_source = pd.concat(parts, ignore_index=True)
by_source.to_parquet(BY_SOURCE_OUT, index=False)

daily = (by_source.groupby(['date', 'ticker'], as_index=False)[['mention_count']].sum())
daily.to_parquet(DAILY_COUNTS_OUT, index=False)
print('combined daily rows:', len(daily), '-> saved', DAILY_COUNTS_OUT)
print('per-source rows    :', len(by_source), '-> saved', BY_SOURCE_OUT)
daily.head()

posts: 2,833,008 | window: 2021-01-01 -> 2021-12-31
by source: {'reddit': 2832937, 'x': 71}
418 word-tickers demoted to cashtag-only (LOAN, EDGE, RENT, ...)
ticker universe size: 12392


combined daily rows: 80931 -> saved C:\Users\alexd\Desktop\GIC\RetailFlow1\data\processed\daily_ticker_counts.parquet
per-source rows    : 80942 -> saved C:\Users\alexd\Desktop\GIC\RetailFlow1\data\processed\daily_ticker_counts_by_source.parquet


,date,ticker,mention_count
0,2021-01-01,AAPL,8
1,2021-01-01,ABBV,1
2,2021-01-01,ABNB,1
3,2021-01-01,ADSK,1
4,2021-01-01,AGG,1


In [4]:
import matplotlib.pyplot as plt
daily['date'] = pd.to_datetime(daily['date'])

# Decide which tickers to draw.
if TICKERS_TO_PLOT:
    chosen = TICKERS_TO_PLOT
else:
    totals = daily.groupby('ticker')['mention_count'].sum().sort_values(ascending=False)
    chosen = list(totals.head(TOP_N).index)
print('plotting:', chosen)

# RAW MENTIONS - every post counts as 1 (combined reddit+x).
plt.figure(figsize=(12, 6))
for ticker in chosen:
    one = daily[daily['ticker'] == ticker].sort_values('date')
    plt.plot(one['date'], one['mention_count'], marker='o', markersize=3, label=ticker)
plt.title('Raw mentions over time (combined)')
plt.xlabel('date'); plt.ylabel('mentions per day')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

plotting: ['GME', 'AMC', 'SNDL', 'PLTR', 'TSLA', 'BB']


C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\1100189491.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Raw mention counts - the actual numbers

Tables of the counts behind the plots: all-time totals per ticker, and each
year's top 10 with the year's total. Useful for sanity-checking (and for
spotting false-positive 'tickers' that are really just all-caps words - most
are auto-demoted by notebook 01's screening now; add stubborn ones to
`BARE_PROSE_STOP` in `src/extract_tickers.py`).

In [5]:
# All-time totals per ticker (top 20 by raw mention count).
alltime = (
    daily.groupby('ticker')
    .agg(total_mentions=('mention_count', 'sum'),
         days_active=('date', 'nunique'),
         busiest_day=('mention_count', 'max'))
    .sort_values('total_mentions', ascending=False)
)
alltime.head(20)

,total_mentions,days_active,busiest_day
ticker,,,
GME,30769,336,4281
AMC,12769,326,1848
SNDL,12251,314,1795
PLTR,8588,354,328
TSLA,8199,363,191
BB,7771,269,1116
CLOV,7716,309,805
NAKD,6950,281,2131
NOK,5049,174,1143


In [6]:
# Top 10 tickers per year, with the raw counts.
daily['year'] = daily['date'].dt.year
per_year = daily.groupby(['year', 'ticker'])['mention_count'].sum().reset_index()
top10 = (
    per_year.sort_values(['year', 'mention_count'], ascending=[True, False])
    .groupby('year')
    .head(10)
    .rename(columns={'mention_count': 'total_mentions'})
)
for year, grp in top10.groupby('year'):
    row = ', '.join(f"{t} ({c:,})" for t, c in zip(grp['ticker'], grp['total_mentions']))
    print(f'{year}: {row}')

2021: GME (30,769), AMC (12,769), SNDL (12,251), PLTR (8,588), TSLA (8,199), BB (7,771), CLOV (7,716), NAKD (6,950), NOK (5,049), SPCE (4,112)


In [7]:
# PER-YEAR GRAPHS - each year picks its own top tickers (raw mentions).
YEARLY_TOP_N = 6

for year in sorted(daily['year'].unique()):
    one_year = daily[daily['year'] == year]
    totals = one_year.groupby('ticker')['mention_count'].sum().sort_values(ascending=False)
    top = list(totals.head(YEARLY_TOP_N).index)
    if not top:
        continue
    plt.figure(figsize=(12, 5))
    for ticker in top:
        one = one_year[one_year['ticker'] == ticker].sort_values('date')
        plt.plot(one['date'], one['mention_count'], marker='o', markersize=2, label=ticker)
    plt.title(f'{year} - top {len(top)} tickers by mentions')
    plt.xlabel('date'); plt.ylabel('mentions per day')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\230986305.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Weekly rolling view + z-scores (combined signal)

Daily counts are jagged (weekday/weekend cycle, small integers). Three
progressively smoother views, all computed on the COMBINED reddit+x signal:

1. **7-day rolling total** - erases the weekend dip entirely.
2. **Rolling z-score** - "how unusual is this week vs the SAME ticker's own
   recent baseline?" `z = (this week - 12-week average) / 12-week std`.
   Makes big and small tickers directly comparable.
3. **Cross-sectional z-score** - "how unusual is this ticker vs ALL OTHER
   tickers this week?" (on log(1+mentions) so mega-tickers don't set the
   scale). Rolling z times a ticker's own take-off; cross-sectional z ranks
   the leaderboard.

**Warm-up caveat:** the rolling z needs `Z_MIN_DAYS` (28) of history, so
the first month of your window has NO z-score. Move `START_DATE` back ~3
months **in notebook 01** if the event you care about is near the start.

In [8]:
# ============ ROLLING / Z-SCORE parameters ============
ROLL_DAYS   = 7      # rolling window for the weekly view (days)
Z_BASELINE  = 84     # rolling z baseline: 84 days = 12 weeks
Z_MIN_DAYS  = 28     # need at least this much history before z is computed
# =======================================================
import numpy as np

wide = (daily.pivot_table(index='date', columns='ticker',
                          values='mention_count', aggfunc='sum')
        .asfreq('D')
        .fillna(0))
print('matrix:', wide.shape[0], 'days x', wide.shape[1], 'tickers')

weekly = wide.rolling(ROLL_DAYS).sum()
roll_mean = weekly.rolling(Z_BASELINE, min_periods=Z_MIN_DAYS).mean()
roll_std  = weekly.rolling(Z_BASELINE, min_periods=Z_MIN_DAYS).std()
zscore = (weekly - roll_mean) / roll_std
logw = np.log1p(weekly)
cs_z = logw.sub(logw.mean(axis=1), axis=0).div(logw.std(axis=1), axis=0)

matrix: 365 days x 5045 tickers


In [9]:
# PLOT 1 - weekly rolling mentions
plt.figure(figsize=(12, 6))
for ticker in chosen:
    if ticker in weekly.columns:
        plt.plot(weekly.index, weekly[ticker], linewidth=1.5, label=ticker)
plt.title(f'{ROLL_DAYS}-day rolling mentions (combined)')
plt.xlabel('date'); plt.ylabel(f'mentions per {ROLL_DAYS} days')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\2472096687.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


In [10]:
# PLOT 2 - rolling z-score: spike vs the ticker's OWN recent history.
plt.figure(figsize=(12, 6))
for ticker in chosen:
    if ticker in zscore.columns:
        plt.plot(zscore.index, zscore[ticker], linewidth=1.2, label=ticker)
plt.axhline(2, color='red', linestyle='--', linewidth=1, label='z = 2 (spike)')
plt.axhline(0, color='black', linewidth=0.6)
plt.title(f'Rolling z-score (vs own {Z_BASELINE}-day baseline)')
plt.xlabel('date'); plt.ylabel('z-score')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\151530347.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


In [11]:
# PLOT 3 - cross-sectional z-score: the ticker vs ALL other tickers that day.
plt.figure(figsize=(12, 6))
for ticker in chosen:
    if ticker in cs_z.columns:
        plt.plot(cs_z.index, cs_z[ticker], linewidth=1.2, label=ticker)
plt.axhline(0, color='black', linewidth=0.6)
plt.title('Cross-sectional z-score (vs all tickers, log scale, same day)')
plt.xlabel('date'); plt.ylabel('z-score across tickers')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\1483373101.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## Reddit vs X - who moves first?

Raw mention counts only, on the window where BOTH sources have data
(the X dumps cover 2015-2020 and Nov 2023+; gap 2021-2023).

1. **Three-line plot** per ticker - Reddit, X, combined - 7-day rolling,
   each divided by its own peak so the SHAPES are comparable.
2. **Lead/lag by cross-correlation**: corr(reddit[t], x[t-k]) for k in
   -MAX_LAG..+MAX_LAG. Best k > 0 = X's moves happened k days BEFORE
   Reddit's (X leads); k < 0 = Reddit leads. Treat +-1 day as simultaneous;
   both series must actually move; this is attention lead/lag, not price.

In [12]:
# ============ COMPARISON PARAMETERS ============
COMPARE_TICKERS  = []   # [] = top TOP_N tickers by combined mentions in the overlap
MAX_LAG          = 14   # days tested in each direction for the lead/lag
MIN_OVERLAP_DAYS = 60   # need at least this many active days to trust the lag
# ===============================================

series = {}   # ticker -> (reddit_series, x_series)

have = set(by_source['source'].unique())
if not {'reddit', 'x'} <= have:
    print('Only', sorted(have), 'in this window - nothing to compare.')
    print('The X data covers 2015-2020 and Nov 2023+: set the TIME WINDOW in')
    print('notebook 01 to overlap those spans.')
else:
    bs = by_source.copy()
    bs['date'] = pd.to_datetime(bs['date'])
    lo = max(bs[bs['source'] == s]['date'].min() for s in ('reddit', 'x'))
    hi = min(bs[bs['source'] == s]['date'].max() for s in ('reddit', 'x'))
    bs = bs[(bs['date'] >= lo) & (bs['date'] <= hi)]
    print('overlap window:', lo.date(), '->', hi.date())

    if COMPARE_TICKERS:
        compare = COMPARE_TICKERS
    else:
        compare = list(bs.groupby('ticker')['mention_count'].sum()
                       .sort_values(ascending=False).head(TOP_N).index)
    print('comparing:', compare)

    def rolling_series(frame, ticker):
        one = frame[frame['ticker'] == ticker]
        if one.empty:
            return None
        s = (one.groupby('date')['mention_count'].sum()
             .reindex(pd.date_range(lo, hi, freq='D')).fillna(0))
        return s.rolling(7, min_periods=1).sum()

    for ticker in compare:
        r = rolling_series(bs[bs['source'] == 'reddit'], ticker)
        x = rolling_series(bs[bs['source'] == 'x'], ticker)
        if r is None or x is None:
            print('skip', ticker, '- no mentions in one of the sources')
            continue
        series[ticker] = (r, x)
        plt.figure(figsize=(12, 4))
        for s, label in ((r, 'reddit'), (x, 'x'), (r + x, 'combined')):
            peak = s.max()
            plt.plot(s.index, s / peak if peak > 0 else s, linewidth=1.4, label=label)
        plt.title(f'{ticker} - 7-day rolling mentions, each line / its own peak')
        plt.xlabel('date'); plt.ylabel('share of own peak')
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\alexd\AppData\Local\Temp\ipykernel_22084\3584131600.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


overlap window: 2021-12-27 -> 2021-12-27
comparing: ['TSLA', 'AAPL', 'NVDA', 'BBIO', 'AMZN', 'MSFT']
skip AAPL - no mentions in one of the sources
skip AMZN - no mentions in one of the sources


In [13]:
# Lead/lag by cross-correlation. corr(reddit[t], x[t-k]) peaking at k=+2
# means what X did two days ago best matches Reddit today -> X leads.
def best_lead_lag(reddit_s, x_s, max_lag=14):
    """Returns (best_k, best_corr). k > 0 = X leads Reddit by k days."""
    best_k, best_c = None, None
    for k in range(-max_lag, max_lag + 1):
        c = reddit_s.corr(x_s.shift(k))
        if pd.notna(c) and (best_c is None or c > best_c):
            best_k, best_c = k, c
    return best_k, best_c

if not series:
    print('nothing to compare - see the cell above')
else:
    rows = []
    for ticker, (r, x) in series.items():
        active_days = int(((r + x) > 0).sum())
        if active_days < MIN_OVERLAP_DAYS:
            rows.append({'ticker': ticker, 'best_lag_days': None, 'corr': None,
                         'verdict': f'only {active_days} active days - too little data'})
            continue
        k, c = best_lead_lag(r, x, MAX_LAG)
        if k is None:
            verdict = 'no correlation (one series may be flat)'
        elif abs(k) <= 1:
            verdict = f'simultaneous (peak corr {c:.2f} at {k:+d}d)'
        elif k > 0:
            verdict = f'X LEADS Reddit by {k} days (corr {c:.2f})'
        else:
            verdict = f'Reddit LEADS X by {-k} days (corr {c:.2f})'
        rows.append({'ticker': ticker, 'best_lag_days': k,
                     'corr': None if c is None else round(c, 3), 'verdict': verdict})
    print(pd.DataFrame(rows).to_string(index=False))

ticker best_lag_days corr                              verdict
  TSLA          None None only 1 active days - too little data
  NVDA          None None only 1 active days - too little data
  BBIO          None None only 1 active days - too little data
  MSFT          None None only 1 active days - too little data
